# **Trial-level stimulus analysis of sentence-final expectancy and semantic integration**

This notebook prepares the stimulus-analysis stage for a revised trial-level reanalysis of the Toffolo et al. (2022) N400 dataset.
The original study tested ERP responses to sentence-final congruent and incongruent words, with effects reported for the Recognition Potential (RP), N400, and Late Positive Component (LPC/P600).
The revised analysis keeps the same sentence-final focus but prepares richer item-level regressors for later linear mixed-effects modelling of ERP amplitudes. The aim is to move from condition-level contrasts toward trial/item-level modelling of continuous linguistic variation across subjects and stimuli.

This notebook belongs to the GitHub repository for the revised Toffolo et al. language reanalysis: [https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised](https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised)

The stimuli-analysis pipeline extracts predictors describing how each final word relates to its preceding sentence context. These include human cloze probability, GPT-2 surprisal, LLM-derived expectancy, human–LLM expectancy disagreement, semantic similarity, lexical frequency, word length, phonology, syntactic complexity, and affective features. This pipeline exports a combined TSV regressor table for later ERP analysis, together with intermediate per-file predictor tables and correlation/VIF diagnostics.
The combined TSV contains the trial/item-level stimulus regressors used to test whether expectancy, semantic fit, lexical properties, phonological structure, syntactic complexity, and affective tone explain variation in RP, N400/PN400, and LPC/P600 amplitudes.

The later ERP analyses use linear mixed-effects models because ERP amplitudes are measured repeatedly across trials, subjects, and stimulus items. This modelling approach allows continuous linguistic predictors to be analysed at the trial level while accounting for variability across participants and stimuli.

The main methodological hypothesis is that trial/item-level linear mixed-effects modelling will provide a cleaner analysis of ERP variability than condition-level averaging. The ERP hypotheses are that low-cloze or high-surprisal sentence endings will increase N400/PN400 amplitude, lexical properties may modulate RP responses, and semantic or syntactic deviations may contribute to LPC/P600 effects. The LLM analysis extends the original cloze-probability approach by comparing human-rated expectancy with model-derived surprisal and identifying where each measure adds explanatory value in the ERP models.

The following cells clone the GitHub repository, install the dependencies listed in `requirements.txt`, unzip `stimuli.zip`, and run `process_stimuli.py`. Running these cells reproduces the combined stimulus-level TSV output and the complementary diagnostic files.

In [1]:
%cd /content

!git clone https://github.com/CPernet/Semantically_Incongruent_or_Congruent_Eggplants_revised.git

%cd /content/Semantically_Incongruent_or_Congruent_Eggplants_revised

/content
Cloning into 'Semantically_Incongruent_or_Congruent_Eggplants_revised'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 283 (delta 58), reused 63 (delta 28), pack-reused 181 (from 1)
Receiving objects: 100% (283/283), 92.75 MiB | 15.88 MiB/s, done.
Resolving deltas: 100% (148/148), done.
/content/Semantically_Incongruent_or_Congruent_Eggplants_revised


In [2]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.7/939.7 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 111.8 MB/s eta 0:00:00


In [3]:
!unzip stimuli/stimuli.zip

Archive:  stimuli/stimuli.zip
 extracting: N400Stimset_cloze-probability-survey_results.tsv  
 extracting: N400Stimset_cloze-probability-survey.tsv  
 extracting: N400Stimset_stimuli_parameters.tsv  
 extracting: stimuli.zip             
 extracting: N400Stimset_cloze-probability-survey.json  
 extracting: N400Stimset_cloze-probability-survey_results.json  
 extracting: N400Stimset_stimuli_parameters.json  
 extracting: README                  
replace README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


**Language Predictor Extraction**

The next stage of the analysis computes the linguistic and psycholinguistic predictors that will later be used as trial-level regressors in the ERP models. The aim is to characterise each sentence-final word using a range of continuous measures describing lexical properties, phonological structure, affective content, contextual predictability, semantic fit, and syntactic complexity.
The following sections compute these predictors step by step and add them to a common dataframe that will ultimately form the stimulus-regressor table used in the later ERP analyses.

In [9]:
from pathlib import Path
import pandas as pd
import spacy

from stimuli_analysis.process_stimuli_stepwise import *
from stimuli_analysis.surprisal import SurprisalModel
from stimuli_analysis.sentence_metrics import SentenceMetrics

file_path = Path("N400Stimset_stimuli_parameters.tsv")

df = load_table(file_path)
prepared = start_processing_table(df, file_path)

out, text_col, target_col, context_col, condition_col = prepared

out = add_sentence_target_context_columns(
    out,
    file_path,
    text_col,
    target_col,
    context_col
)

**Lexical Frequency**

This section computes lexical frequency measures for each target word. The pipeline uses the wordfreq library to estimate both Zipf frequency and raw frequency values for the target word. The resulting measures are added as new predictor columns in the dataframe. Frequency is included because common and rare words are processed differently during language comprehension, making lexical frequency an important control variable when examining contextual and semantic effects.

In [10]:
out = add_lexical_frequency_features(out)

**Phonological Features**

This section computes phonological properties of each target word. Using the CMU Pronouncing Dictionary through the pronouncing package, the pipeline extracts the number of phonemes, the number of syllables, and the onset phoneme for each word. These measures provide information about the sound structure of the target word and are included to capture variation related to word form rather than meaning or contextual predictability.

In [11]:
out = add_phonology_features(out)

**Emotion Features**

This section computes affective properties of the target word. The pipeline queries NRC emotion and VAD lexicons to obtain valence, arousal, dominance, emotion-category labels, and an overall emotionality indicator. These predictors are included because emotional content can influence language processing and neural responses independently of contextual expectancy or semantic fit.

In [12]:
out = add_target_emotion_features(out)

**Surprisal and Sentence Predictability**

This section computes model-based measures of contextual predictability using GPT-2. The pipeline calculates the surprisal of the target word given the preceding sentence context, together with sentence-level surprisal and sentence perplexity measures. Target-word surprisal is computed from the probability assigned by GPT-2 to the observed target word within its context, while sentence-level measures summarise the predictability of the sentence as a whole.

These predictors are included because they provide quantitative estimates of contextual expectation that can later be compared with human cloze probability measures in the ERP analyses.

In [13]:
surprisal_model = SurprisalModel(model_name="gpt2")

out = add_surprisal_features(
    out,
    file_path,
    text_col,
    surprisal_model
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Context–Target Semantic Similarity**

This section computes semantic similarity between the sentence context and the target word. A BERT-based embedding model is used to generate vector representations for the context and target, and cosine similarity is calculated between the resulting embeddings. This predictor provides a model-based estimate of semantic fit between the target word and its preceding context.

In [14]:
semantic_model = SentenceMetrics(model_name="bert-base-uncased")

out = add_semantic_similarity_features(
    out,
    file_path,
    semantic_model
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Syntactic Complexity**

This section computes sentence-level syntactic features using spaCy dependency parses. The pipeline extracts measures including dependency distance, parse depth, token count, and the number of subordinate clauses. These predictors are included to capture variation in sentence structure that may contribute to processing difficulty independently of lexical or semantic factors.

In [15]:
syntax_model = spacy.load("en_core_web_sm")

out = add_syntax_features(
    out,
    file_path,
    text_col,
    syntax_model
)

**Derived Predictors**

This section creates additional predictors from previously computed variables. Z-scored versions of surprisal and semantic similarity are generated, and these standardised measures are combined to create a context-shift index. These derived predictors provide normalised measures that facilitate comparison across variables and summarise relationships between contextual predictability and semantic fit.

In [17]:
out = add_derived_predictor_features(out)

**Cloze Probability Metrics**

This section computes cloze-related predictors using available human and language-model probability estimates. The pipeline identifies cloze-probability columns, calculates human and model unexpectedness measures, derives disagreement metrics between human and model expectations, and creates standardised versions of these predictors. These measures are included because cloze probability is one of the most widely used indicators of contextual expectancy in language-comprehension research.

In [18]:
out = add_final_cloze_features(out)

**Save Final Predictor Table**

This section saves the completed predictor table. The final output contains the original stimulus information together with all computed linguistic and psycholinguistic predictors.

In [19]:
output_dir = Path("language_outputs")
output_dir.mkdir(exist_ok=True)

out.to_csv(
    output_dir / "ALL_language_metrics.tsv",
    sep="\t",
    index=False
)

**Predictor Diagnostics**

This final section evaluates relationships among the computed predictors. Correlation matrices and variance inflation factors (VIFs) are calculated for the main predictor set. These diagnostics help identify redundancy and multicollinearity before the predictors are used in later statistical analyses.

In [20]:
save_predictor_diagnostics(
    out,
    output_dir / "ALL_predictor_diagnostics"
)

In [21]:
!zip -r language_outputs.zip language_outputs

  adding: language_outputs/ (stored 0%)
  adding: language_outputs/ALL_language_metrics.tsv (deflated 71%)
  adding: language_outputs/ALL_predictor_diagnostics_vif.tsv (deflated 40%)
  adding: language_outputs/ALL_predictor_diagnostics_correlations.tsv (deflated 67%)


In [22]:
from google.colab import files
files.download("language_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>